In [ ]:
import numpy as np
import xarray as xr
from matplotlib.cm import RdYlBu_r

from frequensolve.project import Project
from frequensolve.seismic import Acquisition, ReceiverFiber, ReceiverNode
from frequensolve.seismic.layered_model import LayeredModel
from frequensolve.seismic.wavelet import RickerWavelet
from frequensolve.mesh import BoundaryCondition, BoundaryConditionManager
from frequensolve.simulation import (
    Discretization,
    FrequencyDomainJob,
    OutputManager,
    ParaviewOutput,
    SeismicSimulation,
    SolverConfig,
)

project_path = "./scratch/ex_05/"
project = Project(
    name = "rtm",
    pretty_name = "Reverse Time Migration",
    path = project_path,
    load_if_exists = False,
)

# Define Synthetic Simulation

This test uses synthetic data, we'll thus start by defining a simulation with a "sharp" model for generating the data.

## Model setup

In [ ]:
sim = project.new_simulation(
    name = "sharp",
    physics = "coupled",
    dimension = 2,
)
L = 3.0

# New 2D layered model over interval x in [0,1]
model = LayeredModel(
    dimension = 2, 
    x_limits = [0.0, L]
)
Q = 100

model.add_surface(name = "top", z = 0.0)
model.add_layer(
    name = "layer_1",
    properties = { 
        "Vp" : 1.5,
        "Rho": 1.0,
    }
)

x = np.linspace(0.0, L, 251)
surf = xr.DataArray(
    data = (0.03*L - 0.1*(x-L/2) + 0.05*np.sin(x*8*np.pi/L)),
    dims = ["x"],
    coords = {"x": x}
)
model.add_surface(name = "slope", z = surf)
model.add_layer(
    name = "soft_layer",
    properties = { 
        "Vp" : 2.5,
        "Qp" : Q,
        "Rho": 1.4,
        "Vs" : 1.2,
        "Qs" : Q,
    }
)

# x = np.linspace(0.0, L, 251)
# surf = xr.DataArray(
#     data = (0.06 + 0.0125*np.sin(x*8*np.pi/L))*L,
#     dims = ["x"],
#     coords = {"x": x}
# )
# model.add_surface(name = "interface", z = surf)

# model.add_layer(
#     name = "layer_2",
#     properties = { 
#         "Vp" : 3.0,
#         "Qp" : Q,
#         "Rho": 1.8,
#         "Vs" : 1.3,
#         "Qs" : Q,
#         "Epsilon" : 0.4,
#         "Delta" : 0.1,
#     }
# )

surf = xr.DataArray(
    data = (0.15 + 0.05*np.sin(x*4*np.pi/L))*L,
    dims = ["x"],
    coords = {"x": x}
)
model.add_surface(name = "sin", z = surf)

model.add_layer(
    name = "layer_3",
    properties = { 
        "Vp" : 5.2,
        "Qp" : Q,
        "Rho": 2.0,
        "Vs" : 2.6,
        "Qs" : Q,
        "Epsilon" : 0.2,
        "Delta" : 0.1,
    }
)



# x = np.linspace(0.0, L, 251)
# surf = xr.DataArray(
#     data = (0.375*L + 0.4*(x-L/2)),
#     dims = ["x"],
#     coords = {"x": x}
# )
# model.add_surface(name = "flat", z =surf)

# model.add_layer(
#     name = "layer_4",
#     properties = { 
#         "Vp" : 2.5,
#         "Qp" : Q,
#         "Rho": 2.5,
#         "Vs" : 1.25,
#         "Qs" : Q,
#         "Epsilon" : 0.0,
#         "Delta" : 0.0,
#     }
# )

# surf = xr.DataArray(
#     data = (0.4*L - 0.25*(x-L/2)),
#     dims = ["x"],
#     coords = {"x": x}
# )
# model.add_surface(name = "flat", z =surf)

# model.add_layer(
#     name = "layer_5",
#     properties = { 
#         "Vp" : 1.0,
#         "Qp" : Q,
#         "Rho": 2.5,
#         "Vs" : 0.5,
#         "Qs" : Q,
#     }
# )

# surf = xr.DataArray(
#     data = (0.5*L - 0.4*(x-L/2)),
#     dims = ["x"],
#     coords = {"x": x}
# )
# model.add_surface(name = "flat", z =surf)

# model.add_layer(
#     name = "layer_5",
#     properties = { 
#         "Vp" : 3.0,
#         "Qp" : Q,
#         "Rho": 2.5,
#         "Vs" : 2.5,
#         "Qs" : Q,
#     }
# )

# surf = xr.DataArray(
#     data = (0.45*L + 0.4*(x-L/2)),
#     dims = ["x"],
#     coords = {"x": x}
# )
# model.add_surface(name = "flat", z =surf)

# model.add_layer(
#     name = "layer_5",
#     properties = { 
#         "Vp" : 3.5,
#         "Qp" : Q,
#         "Rho": 2.0,
#         "Vs" : 3.5,
#         "Qs" : Q,
#     }
# )
model.add_surface(name = "bottom", z = 0.25*L)
sim += model

# fig, axs = plt.subplots(2,2, figsize = (15,10))
# model.plot("Vp",      aspect = "equal", surfaces = False, ax = axs[0,0])
# model.plot("Vs",      aspect = "equal", surfaces = False, ax = axs[0,1])
# model.plot("Rho",     aspect = "equal", surfaces = False, ax = axs[1,0])
# model.plot("Epsilon", aspect = "equal", surfaces = False, ax = axs[1,1])
# plt.tight_layout()
# plt.show()

## Mesh & Boundary Conditions

In [ ]:
# Define a Hex mesh generator
mesh = model.hex_mesh_generator(n = [8, 4])
sim += mesh

# New boundery condition manager
BCs = BoundaryConditionManager(
    label_type = "geometric"
)
BCs += BoundaryCondition(
    name = "free_surface",
    kind = "free",
    boundaries = ["z_min"]
)
BCs += BoundaryCondition(
    name = "pml",
    kind = "pml",
    boundaries = ["x_min", "x_max", "z_max"],
    pml_wavelengths = 1.0,
    pml_exponent = 3.0,
    pml_reflection = 1e-3,
)
sim += BCs

## Acquisition

For this test we'll use 64 sources; as with receivers this is accomplished by passing 64 source coordinates to the source group below. We'll introduce how to setup and manage more complex acquisitions in example **TODO**.

In [ ]:
acq = Acquisition()

# Sources
# coords = []
# for z in [0.025, 0.15, 0.3, 0.475]:
#     coords.extend([ [x, z] for x in np.linspace(0.0, L, 32) ])
coords = [ [x, 0.015] for x in np.linspace(0.5*L, 0.9*L, 16)]
# coords = [[0, L*0.025]]
# coords = [ [0.5, z] for z in np.linspace(0.0, 0.5, 64)]
if sim.physics == "acoustic" or sim.physics == "coupled":
    acq.add_source_group(
        kind = "scalar",
        coords = coords,
    )
else:
    acq.add_source_group(
        kind = "vector",
        coords = coords,
        direction = [0.0, 1.0],
    )
# acq.add_source_group(
#     kind = "vector",
#     coords = coords,
#     frame = "reference",
#     direction = [0.0, 1.0],
#     domain = 2
# )

# Hydrophones
# if sim.physics == "acoustic" or sim.physics == "coupled":
#     device = ReceiverNode(name = "hydrophone")
#     device.add_component("p","pressure")
# else:
device = ReceiverNode(name = "geophone")
# device.add_component("u_x","velocity",[1., 0.])
device.add_component("u_z","velocity",[0., 1.])

device2 = ReceiverFiber(
    name = "DAS",
    L_gauge = 0.01,
    n_gauge = 100,
    # radius = 0.0003,
    # pitch = 0.001,
)
device2.add_component("eps_tt","strain")

# Surface receivers
coords = [ [x, L*0.13] for x in np.linspace(0.0, L, 251)]
acq.add_receiver_group(
    name = "surface",
    device = device,
    coords = coords,
    frame = "reference",
    domain = 2,
)
sim += acq

## Method, Solver, and Outputs

In [ ]:
# Set adaptivity and order
sim.mesh.set_adapt(min_epw = 4.0, adapt_sources = 0, f_adapt = 10.0)

method = Discretization(order = 4, relaxed_assembly = True)
sim += method

solver = SolverConfig(tolerance = 1.0e-4, grids = 3)
sim += solver

out = OutputManager()
out += ParaviewOutput(
    execute_on = "adapt",
    name = "lsrtm",
    fields = ['pressure', 'velocity', 'stress'],
    properties = ["Vp", "Vs", "Domain", "SubDomain", "pml_stretch"],
    sources = [1, 4, 8, 12, 16], 
    upscale = 0,
    # show_pml = False,
    # post_process = True,
)
sim += out

# Define Smoothed Simulation

Imaging will be done starting from a simulation on a smoothed model. Simulation objects are essentially pointers; we can make a deep copy of the simulation with the `copy` method. We then smooth the model by first sampling it on a uniform (1001x501) grid, and then applying Gaussian smoothing. We'll start with an aggressive smoothing, but then restore the near-surface layer, and smooth again. This is done so that near-surface velocities are accurate, which helps remove primary and surface wave arrivals from the data residual.

In [ ]:
# Copy simulation and smooth aggressively
smoothed = sim.copy(name = "smoothed")

# Smooth model, restore top layer, smooth again
smoothed.model.smooth([501,251], 80)
for key, value in sim.model.layers[0].properties.items():
    if key == "Vp":
       smoothed.model.layers[0].properties[key] = value
# smoothed.model.layers[1].set_property("Vadapt",0.3)
# smoothed.model.plot("Vs", aspect = "equal", surfaces = False, acquisition = smoothed.acquisition, figsize = (4,3))
project += smoothed

project.save()
sharp = project.simulations["sharp"]
smooth = project.simulations["smoothed"]

## Connect to Site and Run

In [ ]:
from frequensolve.orchestrator.sites.hpc import SlurmRunConfig
from frequensolve.orchestrator.sites.local import LocalSite
from frequensolve.orchestrator.sites.stampede3 import Stampede3Site

site = LocalSite(n_workers = 4)
# site = Stampede3Site("problems/ex_05", run_config=SlurmRunConfig(queue="skx-dev", nodes=1, duration="00-02:00:00"))
# site.sync(project)
# allocation = site.attach_allocation()

# Define frequencies used for imaging
f_list = np.linspace(30,30,1)
f_list = f_list + 0.1j

skip = False
# skip = True

job = TimeDomainJob(
    name = "td",
    simulation = sharp,
    f_max = 100,
    T_max = 1.0,
)

# Define job for generating synthetic data
synthetic = FrequencyDomainJob(
    name = "synthetic",
    simulation = sharp,
    f_list = f_list,
)
if not skip:
    site.run(synthetic)

# Define imaging job
rtm = ImagingJob(
    name = "rtm",
    simulation = smooth,
    f_list = synthetic.f_list,
    resolution = [501, 251],
    misfit_norm = "L2",
    images = {
        # "Reflectivity": "up_down",
        "dVp": "FWI:Vp",
        "dVs": "FWI:Vs",
        "p": "pressure"
        # "dRho": "FWI:Rho",
    },
    data_path = synthetic.trace_path,
    regularization = {
        "type": "TV",
        "lambda": 0.5,
        "epsilon": 1.0,
        "iterations": 10,
    },
)
if not skip:
    site.run(rtm)

# Fetch traces and image from the site
syn_db = site.fetch_traces(synthetic, upscale = 4)
fwd_db = site.fetch_traces(rtm, upscale = 4)
image_db = site.fetch_image(rtm)
raw    = image_db.raw_images
images = image_db.smoothed_images
for key in raw.data_vars:
    img = raw[key]
    fig, ax = plt.subplots(figsize = (6, 3))
    img.plot(y = "z", x = "x", yincrease = False, ax = ax, cmap = "Greys")
    plt.show()

    img = images[key]
    fig, ax = plt.subplots(figsize = (6, 3))
    img.plot(y = "z", x = "x", yincrease = False, ax = ax, cmap = "Greys")
    plt.show()
site.close()

# Get Vp from smoothed model (to compute dVp)
mod = smoothed.model.sample_uniform(img.shape[::-1])
Vp = mod.Vp.data.T
dVp = img * Vp

# Get Vp from sharp model (to compute actual dVp)
mod2 = sharp.model.sample_uniform(img.shape[::-1])
Vp2 = mod2.Vp.data.T
dVp2 = (Vp2 - Vp)

A = 3*np.std(np.abs(dVp2.data))
im = plt.imshow(dVp2, vmin=-A, vmax=A, cmap="gray")
plt.colorbar(im, label='Amplitude')
plt.show()

wavelet = RickerWavelet(
    f = 6,
    center = 1/10,
)

# Plot a few traces
A = 0.01
for group in syn_db.groups:
    print(group)
    for shot in [1]:
        for comp in syn_db.components(group):
            td_syn = syn_db.td(group, comp, shot, wavelet, upscale = 4)
            td_fwd = fwd_db.td(group, comp, shot, wavelet, upscale = 4)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
            td_syn.plot(
                vmin = -A,
                vmax = A,
                y = "time",
                yincrease = False,
                cmap = "gray",
                ax = ax1
            )
            ax1.set_title("Synthetic")
            td_fwd.plot(
                vmin = -A,
                vmax = A,
                y = "time",
                yincrease = False,
                cmap = "gray",
                ax = ax2
            )
            ax2.set_title("Forward")
            plt.tight_layout()
            plt.show()